# FIT4112 · LAB 4 · Notebook 03
## Hồi quy dự báo PM2.5 và giải thích kết quả

**FOUNDATION**: dùng cấu hình mẫu cố định, đọc MAE/RMSE, so sánh baseline.  
**APPLIED**: chọn giữa hai bộ đặc trưng bằng validation **nằm trong train**; chỉ đánh giá test một lần sau khi chọn. Cả hai lộ trình cùng rubric và cùng khả năng đạt 100/100.

**Chú ý:** cùng test và cùng độ đo mới so sánh được. One-step rolling không phải fixed-origin multi-step.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab4.py").exists()), None)
assert ROOT is not None, "Mở notebook trong thư mục repo fit4112_lab_4."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.lab4 import (
    LAG_FEATURES, FULL_FEATURES, time_split,
    baseline_predict, error_metrics, fit_rf, model_predict
)
# ===== PARAMETERS =====
TRACK = "FOUNDATION"  # hoặc "APPLIED"; sinh viên TỰ CHỌN lộ trình

path = ROOT / "data/processed/model_frame.csv"
assert path.exists(), "Hãy chạy Notebook 01 và 02 trước."
frame = pd.read_csv(path, parse_dates=["datetime"])
train, test = time_split(frame, test_fraction=0.2)
print("Track:", TRACK, "| train:", len(train), "| test:", len(test))

### 1. Chọn cấu hình mà không nhìn test

Lộ trình Foundation dùng RandomForest và tập lag+hour. Applied có thêm **time validation** nằm trong phần train để lựa chọn một trong hai bộ đặc trưng. **Không** lựa chọn cấu hình dựa trên kết quả cuối của tập test.

In [ ]:
if TRACK == "FOUNDATION":
    chosen_features = list(FULL_FEATURES)
    selection_note = "Cấu hình mẫu Foundation (không tune trên test)."
elif TRACK == "APPLIED":
    fit_part, validation = time_split(train, test_fraction=0.2)
    candidates = {
        "lag_only": list(LAG_FEATURES),
        "lag_plus_hour": list(FULL_FEATURES)
    }
    rows = []
    for name, cols in candidates.items():
        candidate = fit_rf(fit_part, cols)
        val_pred = model_predict(candidate, validation, cols)
        rows.append({"config": name, **error_metrics(validation["y"], val_pred)})
    comparison = pd.DataFrame(rows).sort_values("MAE")
    display(comparison.round(3))
    best_name = comparison.iloc[0]["config"]
    chosen_features = candidates[best_name]
    selection_note = f"Chọn {best_name} theo MAE trên validation nằm trong train."
else:
    raise ValueError("TRACK phải là FOUNDATION hoặc APPLIED.")

print(selection_note)
print("Features:", chosen_features)

### 2. Huấn luyện trên train; so sánh với baseline trên test

MAE và RMSE càng thấp càng tốt. Không yêu cầu mô hình nhất thiết phải thắng persistence; mục tiêu là **đọc và giải thích** kết quả thực nghiệm.

In [ ]:
model = fit_rf(train, chosen_features)
yhat = model_predict(model, test, chosen_features)
baseline = baseline_predict(test)
scores = pd.DataFrame([
    {"model": "Persistence baseline", **error_metrics(test["y"], baseline)},
    {"model": "RandomForest", **error_metrics(test["y"], yhat)}
])
display(scores.round(3))
print(selection_note)

out = ROOT / "data/processed"
preds = pd.DataFrame({
    "datetime": test["datetime"], "y_true": test["y"],
    "y_baseline": baseline, "y_model": yhat
})
preds.to_csv(out / "forecast_predictions.csv", index=False)
with (out / "forecast_metrics.json").open("w", encoding="utf-8") as f:
    json.dump({
        "track": TRACK, "selection": selection_note,
        "features": chosen_features,
        "test_metrics": scores.to_dict(orient="records")
    }, f, ensure_ascii=False, indent=2)
print("Saved outputs in data/processed/")

### 3. Trực quan hóa và diễn giải

Đồ thị giúp chỉ ra các giai đoạn mô hình dự báo kém (đỉnh tăng nhanh, biến động lớn...). Hãy giải thích bằng số liệu, không kết luận dựa vào hình đẹp.

In [ ]:
sample = preds.iloc[:min(120, len(preds))]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sample["datetime"], sample["y_true"], label="Actual", linewidth=1.8)
ax.plot(sample["datetime"], sample["y_baseline"], label="Persistence", alpha=.7)
ax.plot(sample["datetime"], sample["y_model"], label="RandomForest", alpha=.85)
ax.set(title="One-hour-ahead rolling forecast – test subset", ylabel="PM2.5")
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

print("MAE difference (model - baseline):",
      round(float(scores.loc[1, "MAE"] - scores.loc[0, "MAE"]), 3))

### Câu hỏi cần trả lời
1. Ghi MAE/RMSE của baseline và mô hình. Mô hình có vượt baseline không? Lý giải.
2. Vì sao lag_24 có thể hữu ích? Giờ trong ngày là thông tin được biết ở thời điểm dự báo không?
3. Những giờ nào mô hình sai nhiều nhất? Có thể làm gì để cải thiện một cách **hợp lệ**?
4. Vì sao không dùng test để chọn mô hình? Vì sao kết quả trên dữ liệu giả lập không đủ để triển khai thực tế?

**APPLIED**: giải thích thêm bảng validation và bộ đặc trưng đã chọn. **FOUNDATION** có thể đạt điểm tối đa mà không cần bảng này.